In [ ]:
import pandas as pd
import yamls

In [ ]:
users = {}
books = {}
orders = {}

for i in range(1, 6):
    store = f'STORE{i}'
    users[store] = pd.read_csv(f'data/{store}/users.csv')
    orders[store] = pd.read_parquet(f'data/{store}/orders.parquet')

    with open(f'data/{store}/books.yaml') as f:
        books[store] = pd.DataFrame(yaml.safe_load(f))
    books[store].columns = books[store].columns.str.lstrip(':')

In [ ]:
display(users['STORE1'].head())
display(books['STORE1'].head())
display(orders['STORE1'].head())

,id,name,address,phone,email
0,50033,Madelene Hintz III,"Apt. 415 38675 Beau Vista, Faheyville, UT 81337",905.314.5984,bradford@dietrich-homenick.example
1,49617,Mr. Candida Kautzer,"8921 Wilkinson Turnpike, Lake Antoinestad, TX ...",(313) 970-1817,joaquin@murray-smith.example
2,47571,Juan Shields,"Apt. 719 8189 White Knoll, South Pam, AZ 64857",363.251.3408,theresa.kulas@veum-ruecker.test
3,47967,George Streich,"5141 Padberg Vista, Jarrodport, MT 77163-2127",332-256-2176,tuan.lueilwitz@koepp.example
4,49031,Ivy Jones PhD,"673 Jay Inlet, Port Alden, OR 00203",(562) 031-4648,reggie.wilderman@stanton.example


,id,title,author,genre,publisher,year
0,23099,Lawrence of Arabia,Amb. September Walter,Western,Matthias Media,1989
1,22627,Into the Wild,Golda Satterfield,Textbook,Charles Scribner's Sons,2007
2,22817,Jurassic Park,"Miss Lorina Kertzmann, Darron Ernser",Biography/Autobiography,HarperTrophy,1953
3,22730,Minecraft,Nathan Hintz,Narrative nonfiction,Hamish Hamilton,2001
4,22682,Dance Dance Dance,Natalie Schuster,Biography/Autobiography,G-Unit Books,1981


,id,user_id,book_id,quantity,unit_price,timestamp,shipping
0,81642,49038,23163,2,$ 35,"09:28,10-Dec-2025",NULL
1,75869,48585,22545,1,30.0USD,2025-06-25;05:00:01 A.M.,
2,82394,50023,23209,2,37.0 USD,10:31:43 P.M. 2026-01-14,
3,82924,49445,23215,1,52.99USD,03:02:34;17-JUL-2026,
4,78956,49693,22546,1,USD25.25,2025-08-17 12:25,"Apt. 782 35422 Osinski Village, Halvorsonton, ..."


In [ ]:
print(users['STORE1'].dtypes, '\n')
print(books['STORE1'].dtypes, '\n')
print(orders['STORE1'].dtypes)

id         int64
name         str
address      str
phone        str
email        str
dtype: object 

id            int64
title           str
author          str
genre           str
publisher       str
year         object
dtype: object 

id            int64
user_id       int64
book_id       int64
quantity      int32
unit_price      str
timestamp       str
shipping        str
dtype: object


In [ ]:
for name, tables in [('users', users), ('books', books), ('orders', orders)]:
    missing = pd.DataFrame({store: df.isna().sum() for store, df in tables.items()})
    missing.loc['rows'] = [len(df) for df in tables.values()]
    print(name)
    display(missing)

users


,STORE1,STORE2,STORE3,STORE4,STORE5
id,0,0,0,0,0
name,0,0,0,0,0
address,133,130,113,91,112
phone,0,0,0,0,0
email,0,0,0,0,0
rows,3394,3319,3107,3008,3158


books


,STORE1,STORE2,STORE3,STORE4,STORE5
id,0,0,0,0,0
title,0,0,0,0,0
author,0,0,0,0,0
genre,0,0,0,0,0
publisher,2,0,0,8,2
year,2,2,1,3,0
rows,749,770,742,775,750


orders


,STORE1,STORE2,STORE3,STORE4,STORE5
id,0,0,0,0,0
user_id,0,0,0,0,0
book_id,0,0,0,0,0
quantity,0,0,0,0,0
unit_price,0,0,0,0,0
timestamp,0,0,0,0,0
shipping,2580,2238,2248,2617,2365
rows,10493,9181,9280,10877,9714


In [ ]:
for store in users:
    print(store,
          'users:', users[store]['id'].duplicated().sum(),
          'books:', books[store]['id'].duplicated().sum(),
          'orders:', orders[store]['id'].duplicated().sum())

STORE1 users: 0 books: 0 orders: 0
STORE2 users: 0 books: 0 orders: 0
STORE3 users: 0 books: 0 orders: 0
STORE4 users: 0 books: 0 orders: 0
STORE5 users: 0 books: 0 orders: 0


In [ ]:
empty = ['', 'NULL', 'null', 'None', 'nan', '-']

for store in users:
    address = users[store]['address'].str.strip()
    publisher = books[store]['publisher'].str.strip()
    print(store, 'address:', address.isin(empty).sum(), 'publisher:', publisher.isin(empty).sum())

STORE1 address: 30 publisher: 12
STORE2 address: 29 publisher: 14
STORE3 address: 36 publisher: 12
STORE4 address: 30 publisher: 9
STORE5 address: 40 publisher: 9


In [ ]:
for store in books:
    year = books[store]['year'].astype('string')
    bad = year[~year.str.fullmatch(r'\d{4}', na=False)]
    print(store, len(bad), bad.unique().tolist())

STORE1 7 ['', <NA>, '\t']
STORE2 14 ['-1', <NA>, '0', 'NULL', '', '\t', '-']
STORE3 13 ['-', ' ', '', <NA>, '-1', '\t', '0', 'NULL']
STORE4 19 ['', '-', 'NULL', ' ', '0', '\t', '-1', <NA>]
STORE5 10 ['-1', '0', ' ', '-', '']


In [ ]:
all_orders = pd.concat(orders.values())

prices = all_orders['unit_price'].str.replace(r'\d+', '9', regex=True)
print('price formats:', prices.nunique())
prices.value_counts()

price formats: 52


unit_price
$ 9.9      3571
USD 9.9    3558
9.9$       3527
USD9.9     3487
$9.9       3473
9.9 USD    3443
9.9 $      3434
9.9USD     3408
9.9EUR     1590
EUR9.9     1577
€9.9       1552
EUR 9.9    1528
9.9€       1482
$9¢9       1482
9.9 EUR    1481
€9¢9       1475
9.9 €      1430
€ 9.9      1412
9€9¢        523
9$9¢        490
9.$         269
9.USD       266
$9.         266
USD 9.      261
USD 9       259
9 $         259
9. USD      257
USD9.       252
USD9        244
$ 9         240
9USD        235
9$          235
$ 9.        235
9. $        228
$9          217
9 USD       209
9 EUR       125
9EUR        125
EUR 9       117
9€          113
9. €        108
EUR9.       106
€ 9         106
EUR 9.      105
9.EUR       105
EUR9        103
€9          103
9. EUR      101
€9.          98
9 €          98
€ 9.         89
9.€          88
Name: count, dtype: int64

In [ ]:
times = all_orders['timestamp'].str.replace(r'\d', '9', regex=True)
times.value_counts().head(20)

timestamp
9999-99-99T99:99:99        3754
9999-99-99T99:99:99.999    3754
99.99.9999 99:99:99         769
9999-99-99, 99:99           615
9999-99-99;99:99            613
99:99:99;9999-99-99         590
99:99:99,9999-99-99         588
99/99/99 99:99              583
9999-99-99 99:99:99         583
99/99/99,99:99              577
99:99,99/99/99              575
99/99/99,99:99:99           573
99:99:99 99/99/99           571
99/99/99 99:99:99           571
99:99, 99/99/99             569
9999-99-99,99:99            568
99:99:99,99/99/99           566
9999-99-99;99:99:99         563
99:99 99/99/99              563
9999-99-99 99:99            561
Name: count, dtype: int64

In [ ]:
dots = all_orders['timestamp'].str.extract(r'(\d{1,2})\.(\d{1,2})\.(\d{4})').dropna().astype(int)
print('dd.mm.yyyy max:', dots.max().tolist())

slash = all_orders['timestamp'].str.extract(r'(\d{2})/(\d{2})/(\d{2})').dropna().astype(int)
print('mm/dd/yy max:', slash.max().tolist())

dd.mm.yyyy max: [31, 12, 2026]
mm/dd/yy max: [12, 31, 26]


In [ ]:
store1 = users['STORE1']
print(store1['phone'].str.replace(r'\d', '9', regex=True).value_counts())
print('same email:', store1['email'].duplicated().sum())
print('same name:', store1['name'].duplicated().sum())

phone
(999) 999-9999    864
999.999.9999      859
999-999-9999      853
999 999 9999      818
Name: count, dtype: int64
same email: 129
same name: 121
